In [ ]:


import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder

#import tensorflow_datasets as tfds

In [ ]:
#path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [ ]:
# Sourced directly from TruckerPath
park_data_1 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_1 - Copy.csv")
park_data_2 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_2 - Copy.csv")
park_data_3 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_3 - Copy.csv")
park_data = pd.concat([park_data_1, park_data_2, park_data_3], ignore_index=True)

In [ ]:
truck_stop = pd.read_excel("output_excel\Model_Stops_V3.xlsx")

In [ ]:
avail_park = park_data[park_data["pin id"].isin(truck_stop["pin id"].unique())].copy()

In [ ]:
avail_park["time(utc)"] = pd.to_datetime(avail_park["time(utc)"])

In [ ]:
avail_park["day_of_week"] = avail_park["time(utc)"].dt.dayofweek
avail_park["day_name"] = avail_park["time(utc)"].dt.day_name()
avail_park["hour"] = avail_park["time(utc)"].dt.hour
avail_park["month"] = avail_park["time(utc)"].dt.month

In [ ]:
avail_park = avail_park[avail_park["parking status"] != 'Paid'].copy()

In [ ]:
reg_df = avail_park[avail_park['pin id'] == '0583a5874b6f5b0b792d133c942ef95c']

In [ ]:
reg_df

In [ ]:
reg_df = reg_df[['parking status', 'day_of_week', 'hour']].copy()

In [ ]:
## Split into train and test sets 80% train, 20% test.
num_training_examples = int(0.8 * len(reg_df))
train = reg_df[:num_training_examples]
test = reg_df[num_training_examples:]

print('The training data has a shape of:', train.shape)
print('The test data has a shape of:', test.shape)

In [ ]:
## categorical and numeric columns
categorical = ["parking status", "day_of_week", "hour"]

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
train_categorical = train[categorical]
encoded = encoder.fit_transform(train_categorical)
encoded_train_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(train_categorical.columns))
test_categorical = test[categorical]
# encoded_test = encoder.fit_transform(test_categorical)
encoded_test_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(test_categorical.columns))


In [ ]:
encoded_train_df.columns

In [ ]:
# ## Encode categorical data (Ordinal Encoding)
# train_categorical = train[categorical]
# train_categorical = train_categorical.apply(lambda x: LabelEncoder().fit_transform(x), axis = 0)
# test_categorical = test[categorical]
# test_categorical = test_categorical.apply(lambda x: LabelEncoder().fit_transform(x), axis = 0)
# train_categorical

In [ ]:
# Separate features (X) from labels (y) for both training and testing sets
train_X = encoded_train_df.drop(['parking status_Full', 'parking status_Lots', 'parking status_Some'],
                                axis=1)  # Drop the 'Loan_Status' column from training data
train_y = encoded_train_df[['parking status_Full', 'parking status_Lots',
                            'parking status_Some']]  # Keep only the 'Loan_Status' column as labels for training
test_X = encoded_test_df.drop(['parking status_Full', 'parking status_Lots', 'parking status_Some'],
                              axis=1)  # Drop the 'Loan_Status' column from test data
test_y = encoded_test_df[['parking status_Full', 'parking status_Lots',
                          'parking status_Some']]  # Keep only the 'Loan_Status' column as labels for testing

In [ ]:
## Define the model
nn = tf.keras.Sequential([
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')

])

In [ ]:
## Train the model
nn.compile(loss='categorical_crossentropy',  # Loss function for binary classification
           optimizer='adam',  # optimizer to use
           metrics=['accuracy'])  # Evaluation metrics to use for training

# fit the model to the training data
nn_history = nn.fit(train_X, train_y, epochs=50, verbose=True)

In [ ]:
## Evaluate the model
train_loss, train_acc = nn.evaluate(train_X, train_y, verbose=False)
test_loss, test_acc = nn.evaluate(test_X, test_y, verbose=False)
print("The training accuracy is:", train_acc)
print("The testing accuracy is:", test_acc)

In [ ]:
## Obtaining probability predictions with the trained model (on the first 5 individuals in the training dataset)
train_X_first5 = train_X.head(5)
print(train_X_first5)
predictions = nn.predict(train_X_first5)
predictions



In [ ]:
train_X_first5 = test_X

predictions = nn.predict(train_X_first5)
predicted_classes = np.argmax(predictions, axis=1)  # pick class with highest probability

print("Accuracy:", accuracy_score(test_y, predicted_classes))
print(classification_report(test_y, predicted_classes))
